# Generating slip on a SCEC CFM fault plane — Step 2: remesh, set up, and run k223d

This continues from
[`01_cfm_to_k223d_load_and_inspect.ipynb`](01_cfm_to_k223d_load_and_inspect.ipynb),
which downloaded and parsed the CFM 6.1 representation of the San Bernardino Mountains
section of the San Andreas fault.

From here the workflow is: remesh the raw CFM triangulation to a target resolution,
assign a rupture velocity and nucleation point, flag the free surface, then run k223d
and write the output.

```
points, cells  ─→  remesh_fault  ─→  k223d
```

## Set parameters and subroutines used in notebook

In [12]:
import numpy as np
import pathlib
import sys

# Mesh wrangling
import gmsh
import meshio

# convert from geographical coordinates
import pyproj

# plotting mesh
import plotly.graph_objs as go

In [13]:
kilo = 1000.

### Custom Python modules

In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
folder_path = '../../../local_py_scripts/'
sys.path.append(folder_path)
from plotting import *
from notebook_utils import *
# from fault_scaling import *
import scaling_relations as scaling

import k223d

In [16]:
%reload_ext autoreload

## Import the CFM fault

Step 1 left the parsed surface cached as a legacy ASCII VTK file in `cfm_cache/`
(`write_vtk_fields` in step 1 wrote it precisely so gmsh could reload it without
re-parsing the raw GOCAD `.ts`). Rather than re-running the download we reload the
mesh straight from that cache with `extract_mesh` (from `notebook_utils`, shared with
every other notebook in this repo).

In [17]:
CACHE = pathlib.Path("cfm_cache")
FAULT = "SBMT_San_Andreas_CFM6_500m.vtk"

x, y, z, cells = extract_mesh(str(CACHE / FAULT))
points = np.column_stack([x, y, z])

ncells = len(cells)
nnodes = len(points)
print("No of Nodes :   ", nnodes)
print("No of cells:  ",  ncells)

Info    : Reading 'cfm_cache/SBMT_San_Andreas_CFM6_500m.vtk'...
Info    : Reading 10414 points
Info    : Reading 20271 polygons
Info    : Done reading 'cfm_cache/SBMT_San_Andreas_CFM6_500m.vtk'
number of nodes      10414
number of elements     20271
No of Nodes :    10414
No of cells:   20271


### A note on coordinates

**CFM is already projected** — UTM zone 11, NAD27 (EPSG:26711), metres — so
there is nothing to do here and `points` can go straight into the size calculations.

NAD27 is *not* WGS84; treating one as the other displaces things by of order 100 m in southern
California. So rather than `pyproj.Proj(proj='utm', zone=11, ellps='WGS84')`, build an
explicit transformer between the two coordinate reference systems:

In [18]:
# CFM native CRS -> WGS84 lon/lat, used at the end of the workflow
CFM_CRS = pyproj.CRS.from_epsg(26711)     # UTM zone 11N, NAD27
GEO_CRS = pyproj.CRS.from_epsg(4326)      # WGS84 lat/lon
to_geo  = pyproj.Transformer.from_crs(CFM_CRS, GEO_CRS, always_xy=True)

lon, lat = to_geo.transform(points[:, 0], points[:, 1])
print(f"fault spans {lon.min():.3f} to {lon.max():.3f} E, "
      f"{lat.min():.3f} to {lat.max():.3f} N")

fault spans -117.785 to -116.581 E, 33.947 to 34.417 N


In [ ]:
plot_mesh(cells, points)

## Cell size on the current mesh

Using the shared helpers from `notebook_utils`.

In [22]:
tri_area = tri_area_3d(points[cells])
ave_cell_area = np.mean(tri_area)
total_area = np.sum(tri_area)
edge_dist = tri_dist(points[cells])
ave_dist = np.mean(edge_dist) / kilo

print("total size of fault surface:   ", total_area / kilo**2, 'km^2')
print("total number of cells:   ", ncells)
print("mean distance along side of cell is:   ", ave_dist, 'km')

total size of fault surface:    2414.4565625229434 km^2
total number of cells:    20271
mean distance along side of cell is:    0.5264703260455529 km


### How large an earthquake can this fault host?

**Pick one appropriate to the faulting style.** The San Bernardino Mountains section is a continental strike-slip fault, so a Wells & Coppersmith or Leonard relation is a more natural choice; Strasser is calibrated on subduction interface events and will not be right here. Scaling relations are provided by `scaling.compute(relation, category, magnitude=... or area=..., area_unit=...)`; run `scaling.available_relations()` to see what relations/categories your copy provides, and set `RELATION` below accordingly.

In [23]:
print(scaling.available_relations()) 

{'wells_coppersmith_1994': ['strike-slip', 'reverse', 'normal', 'all'], 'strasser_2010': ['interface', 'intraslab'], 'leonard_2010': ['strike-slip', 'dip-slip', 'interplate'], 'hanks_bakun_2002': ['strike-slip'], '_illustrative': ['example_both', 'example_area_only']}


In [24]:
RELATION = "wells_coppersmith_1994"  # <-- set to a strike-slip / continental relation, see scaling.available_relations()

max_mag = scaling.compute(RELATION, "strike-slip", area=total_area, area_unit="m2")
print(f"the largest earthquake possible on fault (Mw):   {max_mag :.2f}")

the largest earthquake possible on fault (Mw):   7.43


In [25]:
Mw = 7.4
target_area = scaling.compute(RELATION, "strike-slip", magnitude=Mw, area_unit="m2")
print(f"Area of earthquake:   {target_area / kilo**2 :.2f} km^2")
print(f"total size of fault surface:   {total_area / kilo**2 :.2f} km^2")
print(f"mean cell area:    {ave_cell_area / kilo**2  :.3f} km^2")
print(f"No. of cells for rupture {int(target_area / ave_cell_area)}")

Area of earthquake:   1737.80 km^2
total size of fault surface:   2414.46 km^2
mean cell area:    0.119 km^2
No. of cells for rupture 14590


The number of cells the rupture lands on is what controls how well the stochastic slip
patches are resolved. Note `rmin` in k223d defaults to 2.0, meaning the smallest
sub-event radius is twice the mean cell radius — so a few thousand cells inside the
rupture is a sensible target, and much fewer starts to under-resolve the cascade.

In [26]:
target_no_cells = 10000
target_cell_area = (target_area / target_no_cells)
print('Aim to have cells with the following average area of.....:',
      target_cell_area / kilo**2, 'km^2')
print('The current average area is ....:', ave_cell_area / kilo**2, 'km^2')

Aim to have cells with the following average area of.....: 0.17378008287493762 km^2
The current average area is ....: 0.11910890249730866 km^2


Assuming the triangles are equilateral, the area of a triangle of edge length $a$ is

$$A = \frac{\sqrt{3}}{4}\,a^{2}$$

where $A$ is the triangle area and $a$ is the edge length. Rearranging for the edge
length gives the target we hand to gmsh:

$$a = \sqrt{\frac{4A}{\sqrt{3}}}$$

In [27]:
target_edge = np.sqrt(target_cell_area * 4. / np.sqrt(3))
print('This is the target average length of a cell edge.....', int(target_edge), 'm')

This is the target average length of a cell edge..... 633 m


## Refine mesh in Gmsh

`remesh_fault` (in the shared `notebook_utils`, used by every notebook in this repo)
rebuilds the surface geometry from the triangulation and meshes it to a uniform target
edge length.

**One caution specific to this fault.** The CFM surface has 20 271 triangles. Building
the geometry as one plane surface per input triangle (an earlier approach this
notebook used to use) would create on the order of 60 000 lines and 20 000 surfaces
for a mesh this size, and could only ever refine, not coarsen. `remesh_fault` instead
passes the triangulation to gmsh as a single discrete surface in one go, and
`classifySurfaces` / `createGeometry` reparametrise it. Cost is essentially
independent of the input triangle count, and it can coarsen as well as refine.

In [28]:
# remeshing fault; write_files=True also saves 'refined_mesh.vtk' (viewable in
# paraview) and 'refined_mesh.msh' (readable by gmsh) -- set False to skip that
new_x, new_y, new_z, new_cells = remesh_fault(
    points, cells, target_edge, 'refined_mesh', write_files=True)

new_points = np.transpose([new_x, new_y, new_z])
n_ncells = len(new_cells)

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    : Classifying surfaces (angle: 180)...
Info    : Splitting triangulations to make them parametrizable:
Info    : Model has 0 non manifold mesh edges and 555 boundary mesh edges
Info    : Found 1 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.31946s, CPU 0.321045s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 3.7409e-05s, CPU 2.8e-05s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.136507s, CPU 0.137191s)                             
Info    : Meshing 1D...
Info    : Meshing curve 3 (Discrete curve)
Info    : Done meshing 1D (Wall 0.0102445s, CPU 0.010375s)
Info    : Meshing 2D...
Info    : Meshing surface 2 (Discrete surface, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.419179s, CPU 0.419848s)
Info    : 7438 nodes 

## Check new cell size against target

In [29]:
print("No of nodes in new mesh:      ", len(new_points))

tri_area = tri_area_3d(new_points[new_cells])
print("mean cell area:   ", np.mean(tri_area) / kilo**2, 'km^2')
print('No. of cells for rupture:', int(target_area / np.mean(tri_area)))

No of nodes in new mesh:       7438
mean cell area:    0.1671272283287397 km^2
No. of cells for rupture: 10398


### Did the remesh preserve the surface?

Reparametrisation can quietly move the surface — patches can be smoothed across sharp
bends, or a badly chosen classification angle can weld parts of the fault together.
Two checks are worth the few seconds they cost.

The first is the total area, which should be within a fraction of a percent. The second
is the topology, which should still be a single manifold sheet with the same Euler
characteristic as the input.

In [30]:
new_total_area = np.sum(tri_area)
print(f"area before / after : {total_area/kilo**2:.1f} / "
      f"{new_total_area/kilo**2:.1f} km^2  "
      f"({100*(new_total_area-total_area)/total_area:+.2f} %)")

mesh_report(new_points, new_cells, "remeshed")

area before / after : 2414.5 / 2413.5 km^2  (-0.04 %)
--- remeshed ------------------------------------------------
  vertices / edges / triangles : 7438 / 21878 / 14441
  Euler characteristic V-E+F   : 1
  boundary edges               : 433
  non-manifold edges           : 0
  connected components         : 1
  total area                   : 2413.5 km^2
  triangle edge scale (sqrt A) : min 244 m, median 414 m, max 525 m
  x range                      : 427973 to 538680 m
  y range                      : 3756099 to 3808466 m
  z range                      : -16997 to 2063 m


array([210201.06788305, 196038.93278087, 210308.48379375, ...,
        77981.94936653, 124532.88306005, 105944.60115729])

In [ ]:
# mesh1 (red) is the original CFM mesh, mesh2 (black) the remeshed one
compare_meshes(cells, points, new_cells, new_points)

## Add rupture velocity to mesh

Rupture velocity increases with depth:

$$v_i = v_0 + g\,|\bar{z}_i|$$

where $v_i$ is the rupture velocity assigned to cell $i$, $v_0$ is the velocity at the
free surface, $g$ is the depth gradient, and $\bar{z}_i$ is the mean elevation of the
three nodes of cell $i$ (negative below sea level, hence the absolute value).

In [32]:
rupt_vel  = 2000.   # Set the initial velocity at the surface [m/s]
rupt_grad = 0.25    # Set the depth dependent velocity gradient [m/s per m]

In [33]:
# set velocity for each cell
cell_vel = np.zeros(n_ncells)
for i in range(n_ncells):
    c_id = new_cells[i]
    mean_depth = np.abs(np.mean(new_z[c_id]))
    cell_vel[i] = rupt_vel + rupt_grad * mean_depth

In [ ]:
plot_mesh_cell(new_cells, new_points, cell_vel, 'velocity')

## Assign surface rupture

Step 1 reported an elevation range of **-17 019 m to +2 064 m**. The positive values
are not an error: CFM hangs its fault surfaces from the *topographic* surface, not
from sea level, and the San Bernardino Mountains rise well above it. So flagging every
node above some fixed elevation (say, -1500 m) would be measuring from the wrong
reference — a node at +1 800 m in the mountains and a node at -200 m in a valley are
both at the free surface, but a fixed cut-off treats them very differently and picks
up a band of genuinely buried nodes wherever the topography is high.

A more robust rule is topological: find the nodes on the *boundary* of the mesh (edges
belonging to exactly one triangle), then keep the boundary nodes that are shallower
than the mid-depth of the fault. Those are the upper tip line; the rest of the boundary
is the bottom and the two lateral ends. `find_surface_nodes` (in `notebook_utils`)
implements this, and works regardless of where sea level sits relative to the fault.

In [35]:
print('fault depth range (m):    ', np.min(new_z), 'to ', np.max(new_z))

fault depth range (m):     -16997.114371919255 to  2063.4709761406316


In [36]:
surface_nodes = find_surface_nodes(new_points, new_cells, depth_fraction=0.7)
print('number of surface nodes.....', np.sum(surface_nodes))
print('their elevation range (m)...',
      new_z[surface_nodes == 1].min(), 'to', new_z[surface_nodes == 1].max())

number of surface nodes..... 185
their elevation range (m)... -597.096955125192 to 2063.4709761406316


In [ ]:
plot_highlight_nodes(new_cells, new_points, surface_nodes, 'surface nodes')

Inspect that plot carefully before moving on. The highlighted nodes should trace the
fault trace along the top edge only. If they wrap around the lateral ends of the fault
as well, tighten the `depth_fraction` argument; if the trace is broken, loosen it.

### Nucleation location

`assign_nucleation_location` expects the epicentre in the **mesh's own coordinates** —
UTM zone 11 / NAD27 metres here. If you have a candidate hypocentre in lon/lat, project
it with the transformer built earlier, running it in the forward direction.

The example below simply nucleates near the southeastern end of the fault so the
rupture propagates along strike. Change `nuc_lon`, `nuc_lat` to place it elsewhere, and
check that `assign_nucleation_location` does not error because the point falls outside
the mesh.

In [38]:
# to_utm = pyproj.Transformer.from_crs(GEO_CRS, CFM_CRS, always_xy=True)

# nuc_lon, nuc_lat = -116.85, 33.95        # <-- set your hypocentre here if you have geographical coordinates
# quake_x, quake_y = to_utm.transform(nuc_lon, nuc_lat)
# print(f"nucleation at {quake_x:.0f}, {quake_y:.0f} (UTM11N/NAD27)")


quake_x, quake_y = 512.0577*kilo,3771.376*kilo
itime = assign_nucleation_location(quake_x, quake_y,
                                   new_x, new_y, new_z, new_cells, cell_vel)

Source is in cell 12133
Initial travel time on starting cell edges: [0.0337704  0.12458957 0.15690287]


In [ ]:
plot_mesh_node(new_cells, new_points, itime, 'time')

## Run k223d
In this case we assume that the slip pdf is uniform 

In [40]:
slip,_ = k223d.compute_source(new_points,new_cells, mw=Mw )


 build_mesh_geometry


Total fault area:        2.4135E+09 m^2
Estimated fault width:   25173.97 m
 calculate distance to border
 calculate distance between all nodes


 construct slip distribution


 using max of pdf fn for location of first subevent 
total slipping area        2.4135E+09
average slip:                2.19 m
moment of event            1.5849E+20
Moment magnitude of event    7.40


In [ ]:
plot_mesh_cell(new_cells,new_points,slip,'slip (m)')

### Run k223d 
This time to acquire rupture history 

In [42]:
slip,rupt_time = k223d.compute_source(new_points,new_cells, mw=Mw ,velocity=cell_vel,time=itime,surface_nodes=surface_nodes)


 build_mesh_geometry


Total fault area:        2.4135E+09 m^2
Estimated fault width:   25173.97 m
 calculate distance to border
 calculate distance between all nodes


 construct slip distribution


 using max of pdf fn for location of first subevent 


 calculate rupture time
 starting trilateration.....


total slipping area        2.4135E+09
average slip:                2.19 m
moment of event            1.5849E+20
Moment magnitude of event    7.40


In [ ]:
plot_mesh_cell_node(new_cells, new_points, slip,rupt_time, 'Slip (m)', contours=True, n_contours=10)

## Output mesh for k223d

Save all fields to a vtk file that can be read by _Paraview_.

In [ ]:
write2fmt(new_points, new_cells, 'output', fmt='vtk',
          velocity=cell_vel,slip=slip, time=rupt_time)

## Where this leaves us

Starting from the raw CFM triangulation, this notebook has:

- remeshed it to a target resolution with `remesh_fault`,
- assigned a depth-dependent rupture velocity,
- flagged the free surface topologically with `find_surface_nodes`,
- picked a nucleation point and run `k223d.compute_source` (both with a uniform slip
  pdf, and with rupture-time-dependent slip),
- computed slip distribution and rupture time 
- outputted results to a vtk file

To adapt this to another CFM fault, change `FAULT` near the top of this notebook (and
re-run step 1 if it is not already cached), then re-run from there — everything else,
including the size targets and the topological surface-rupture rule, is generic.